# sketch 컬러 증강 파인튜닝 (저비용 검증) — S15P11B209-754

흑백 선화로 학습된 sketch_base를 **컬러/색칠에 강건**하게 만들기 위한 저비용 검증.
기존 v4 흑백 데이터셋을 **랜덤 색으로 재색칠**(색=객체 무관, 모양으로만 학습)해 `sketch_base_v4.pt`에서
소수 epoch만 파인튜닝한다. 전체 재학습(~10h) 전에 효과부터 확인.

**JupyterHub에서 위→아래 Run All.** 필요: `htp_yolo/stageA_quickdraw`(v4 데이터셋) + `sketch_base_v4.pt`.
검증된 로직(`finetune_color_aug.py`의 `recolor`·`build_augmented`)을 그대로 import해 쓴다.

In [ ]:
import sys, random
from pathlib import Path

# 같은 폴더의 finetune_color_aug.py 를 import (JupyterHub는 노트북 폴더를 경로에 넣지만 안전하게 보강).
sys.path.insert(0, str(Path.cwd()))
import finetune_color_aug as fca

# ── config: 필요하면 여기서 바꾸세요 ──
fca.EPOCHS = 10          # 저비용 검증용 소수 epoch (빠르게: 5)
fca.IMG_SIZE = 960
fca.BATCH = 16
fca.KEEP_LINE_RATIO = 0.5  # 선화 유지 비율(나머지는 색 입힘)
fca.FILL_PROB = 0.5        # 색 입힐 때 '영역 채움'(색칠)까지 넣을 확률

print('원본 데이터셋:', fca.SRC_DATASET, '| 존재:', fca.SRC_DATASET.exists())
print('v4 가중치   :', fca.BASE_WEIGHTS, '| 존재:', fca.BASE_WEIGHTS.exists())
print('증강본 출력 :', fca.AUG_DATASET)
print('결과 가중치 :', fca.OUT_WEIGHTS)

## 1. 증강 미리보기 (학습 전 눈으로 확인)

같은 그림이 랜덤 색·랜덤 색칠로 바뀌되 **형태(획)는 보이는지** 확인한다.

In [ ]:
import cv2, numpy as np
import matplotlib.pyplot as plt

srcimg = sorted((fca.SRC_DATASET/'train'/'images').glob('*.jpg'))[:3]
assert srcimg, f'원본 학습 이미지가 없습니다: {fca.SRC_DATASET}/train/images'
r = random.Random(7)
fig, ax = plt.subplots(len(srcimg), 4, figsize=(16, 4*len(srcimg)))
ax = ax.reshape(len(srcimg), 4)
for i, p in enumerate(srcimg):
    orig = cv2.imread(str(p))
    cols = [orig] + [fca.recolor(orig, r) for _ in range(3)]
    for j,(a,im) in enumerate(zip(ax[i], cols)):
        a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.axis('off')
        a.set_title('original' if j==0 else f'recolor {j}')
plt.tight_layout(); plt.show()

## 2. 파인튜닝 실행 (증강 데이터셋 생성 → v4에서 파인튜닝 → v5_ft.pt 저장)

`fca.main()`이 재색칠 데이터셋 생성 + 학습 + 저장을 한 번에 한다. GPU면 자동 사용.

In [ ]:
fca.main()

## 3. 검증 — v4 vs v5 파인튜닝 비교

`test_images`(색선·색칠)에서 v4와 v5_ft를 나란히 탐지해 오탐↓·정탐↑를 본다.

In [ ]:
from ultralytics import YOLO
conf = fca.__dict__.get('YOLO_CONF', 0.2)
m_v4 = YOLO(str(fca.BASE_WEIGHTS))
m_v5 = YOLO(str(fca.OUT_WEIGHTS))

def summ(model, img):
    r = model.predict(img, conf=conf, verbose=False)[0]
    return r, sorted([(r.names[int(c)], round(float(p),2)) for c,p in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist())], key=lambda d:-d[1])

test_dir = Path('test_images')
tests = sorted(test_dir.glob('*.png')) if test_dir.exists() else []
for p in tests:
    img = cv2.imread(str(p))
    if img is None: continue
    r4,d4 = summ(m_v4, img); r5,d5 = summ(m_v5, img)
    print(f'[{p.name}]  v4 {len(d4)}건 {d4}')
    print(f'{" "*(len(p.name)+3)}  v5 {len(d5)}건 {d5}')
    fig, ax = plt.subplots(1,2, figsize=(14,7))
    ax[0].imshow(cv2.cvtColor(r4.plot(), cv2.COLOR_BGR2RGB)); ax[0].set_title(f'v4 ({len(d4)})'); ax[0].axis('off')
    ax[1].imshow(cv2.cvtColor(r5.plot(), cv2.COLOR_BGR2RGB)); ax[1].set_title(f'v5_ft ({len(d5)})'); ax[1].axis('off')
    plt.tight_layout(); plt.show()